# AI SkyRun · Tier 1 — навчи дрон літати

Дрон **не знає карти**. Він бачить лише **номер клітинки**, в якій стоїть.
Кожен епізод — спроба долетіти A → B. За досвідом він оновлює оцінку цінності станів.

Через десятки-сотні спроб зʼявляється функція цінності `V(s)`, а `−V(s)` — це
потенціальна поверхня, по якій дрон «скочується» до цілі.

> **Правило гри:** доступу до координат дерев **немає**. Тільки досвід.
> Якщо ви десь написали `env.trees` — це вже картографування, а не RL.

Ваша робота — **дві дірки `# TODO`** нижче: правило Беллмана й дизайн винагороди/тюнінг.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent.parent))

import numpy as np
import matplotlib.pyplot as plt

from tier_d.env.gridworld import GridWorld
from tier_d.env.constants import N_ACTIONS
from tier_d.oracle.astar import optimal_length      # еталон оптимуму L*
from tier_d.scoring.seeds import PUBLIC_SEEDS

SEED, N_TREES = 0, 16
env = GridWorld(N_TREES, SEED)
LSTAR = optimal_length(env.trees)            # оракул бачить дерева — ви ні

print(f"станів: {env.n_states}   дій: {env.n_actions}")
print(f"оптимум оракула L* = {LSTAR:.3f}")

## Що дає середовище

| | |
|---|---|
| `s = env.reset()` | номер клітинки старту (ціле число) |
| `s2, r, done, info = env.step(a)` | `a ∈ 0..7` — 8 напрямків |
| винагорода | `−1` крок · `−100` колізія (кінець) · `+100` ціль (кінець) |
| `info["collision"]`, `info["goal"]` | справжні термінали |
| `info["truncated"]` | скінчився ліміт кроків — **це не термінал** |

Різниця між `terminal` і `truncated` важлива: у термінал **не бутстрепимо**,
у truncated — бутстрепимо. Плутанина тут тихо ламає навчання.

## TODO 1 — правило Беллмана

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\underbrace{r + \gamma \max_{a'} Q(s',a')}_{\text{ціль}} - Q(s,a)\big]$$

У терміналі майбутнього немає: ціль = просто `r`.

In [ ]:
def q_update(Q, s, a, r, s2, terminal, alpha, gamma):
    """Оновити Q[s, a] на місці. Повернути TD-помилку (для дебагу)."""
    # TODO(1): порахуйте ціль (target)  ← ваша логіка: термінал = кінець гри (просто r),
    #          звичайний крок = гра триває (додаємо обіцянку майбутнього gamma*max(Q[s2]))
    if terminal:
        target = r
    else:
        target = r + gamma * max(Q[s2])

    # TODO(2): зробіть крок до цілі  ← ваша формула: Q[s,a] = Q[s,a] + alpha * δ  (δ = td)
    td = target - Q[s, a]
    Q[s, a] = Q[s, a] + alpha * td
    return td

## TODO 2 — розріджена винагорода і reward shaping

**Головна пастка.** Приз лише в цілі. Випадковий агент на сітці 25×25 може
**дуже довго** не дійти до B жодного разу → цінності нема звідки поширюватись →
«нічого не вчиться», без жодної помилки в коді.

Ліки — **потенціал-орієнтований shaping** (Ng–Harada–Russell):

$$F(s,s') = \gamma\,\Phi(s') - \Phi(s), \qquad \Phi(s) = -\,\text{dist}(s, B)$$

Теорема гарантує: **оптимальна політика не змінюється**. Shaping лише пришвидшує.

Одна тонкість: у справжньому терміналі $\Phi(s') := 0$. Інакше гарантія зникає.

> Увімкніть `USE_SHAPING = True`, якщо агент не вчиться. Це не читерство — це частина кору.

In [ ]:
from tier_d.env.constants import CELL, GRID_N
from tier_d.env.occupancy import GOAL_CELL

USE_SHAPING = True          # ← спробуйте False і подивіться, що станеться

def potential(s):
    """Φ(s) = −відстань від клітинки s до цілі (у світових одиницях)."""
    r, c = divmod(s, GRID_N)
    gr, gc = GOAL_CELL
    return -np.hypot(r - gr, c - gc) * CELL

def shaping_F(s, s2, gamma, terminal):
    if not USE_SHAPING:
        return 0.0
    phi_next = 0.0 if terminal else potential(s2)   # Φ(термінал) = 0
    return gamma * phi_next - potential(s)

## TODO 3 — гіперпараметри

`alpha` темп навчання · `gamma` дисконт · `eps` дослідження (ε-greedy).

ε спадає: спочатку багато випадковості (шукаємо ціль), потім жадібність (шліфуємо шлях).

In [ ]:
ALPHA   = 0.05    # набір A (обережний): менший темп → точніший шлях
GAMMA   = 0.99    # далекоглядний агент (баланс блукання=колізія збережено)
EPS0    = 1.0
EPS_MIN = 0.01    # менше випадкових детурів наприкінці → краща оптимальність
EPISODES = 3000   # на M4 це секунди

In [ ]:
def greedy_rollout(env, Q, max_steps=400):
    """Політ без дослідження: суто argmax. Так вас оцінюватимуть."""
    s = env.reset(); path = [env.xy]; length = 0.0
    for _ in range(max_steps):
        s, _, done, info = env.step(int(np.argmax(Q[s])))
        path.append(env.xy); length += info["moved"]
        if info["collision"]: return dict(success=False, length=length, path=np.array(path))
        if info["goal"]:      return dict(success=True,  length=length, path=np.array(path))
    return dict(success=False, length=length, path=np.array(path))


def train(env, episodes=EPISODES, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, N_ACTIONS))
    curve = []
    decay = int(episodes * 0.6)

    for ep in range(episodes):
        eps = max(EPS_MIN, EPS0 + (EPS_MIN - EPS0) * ep / decay)
        s = env.reset()
        while True:
            a = int(rng.integers(N_ACTIONS)) if rng.random() < eps else int(np.argmax(Q[s]))
            s2, r, done, info = env.step(a)
            terminal = info["collision"] or info["goal"]

            r_train = r + shaping_F(s, s2, GAMMA, terminal)
            q_update(Q, s, a, r_train, s2, terminal, ALPHA, GAMMA)

            s = s2
            if done: break

        if ep % 25 == 0:
            roll = greedy_rollout(env, Q)
            curve.append((ep, roll["length"] if roll["success"] else np.nan))
    return Q, np.array(curve)


Q, curve = train(GridWorld(N_TREES, SEED))
roll = greedy_rollout(GridWorld(N_TREES, SEED), Q)
print(f"дійшов: {roll['success']}   L = {roll['length']:.3f}   L* = {LSTAR:.3f}")
if roll["success"]:
    print(f"оптимальність L*/L = {LSTAR / roll['length']:.3f}   (максимум 1.000)")

## Крива навчання

Якщо лінія так і не зʼявилась — жадібна політика **жодного разу** не дійшла до B.
Це та сама розріджена винагорода. Увімкніть shaping (TODO 2) або підніміть ε.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.axhline(LSTAR, ls="--", color="#8250df", label=f"L* = {LSTAR:.2f}")
ax.axhline(1.15 * LSTAR, ls=":", color="grey", label="поріг 1.15·L*")
ok = ~np.isnan(curve[:, 1])
ax.plot(curve[ok, 0], curve[ok, 1], color="#0969da", lw=1.8, label="жадібна політика")
ax.set_xlabel("епізод"); ax.set_ylabel("довжина шляху"); ax.legend(); ax.grid(alpha=.3)
plt.show()

## Поверхня цінності `−V(s)`

Піки виростають там, де **боляче** — не там, «де дерево».
Дерево, об яке агент жодного разу не вдарився, лишається пласким.

In [ ]:
from tier_d.scaffold.qtools import value_surface
from tier_d.viz.value_surface import plot_value_surface_3d, plot_policy_arrows

e = GridWorld(N_TREES, SEED)
V = value_surface(Q, blocked=e.blocked)     # висота піків = пережита біль

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
plot_value_surface_3d(ax1, V, title="−V(s): виросло з винагороди")
ax2 = fig.add_subplot(1, 2, 2)
plot_policy_arrows(ax2, Q, e.blocked, e.trees, roll["path"], title="жадібна політика")
plt.show()

## Самоперевірка на public seeds

Фінал рахується на **прихованих** seed-ах, яких ви не бачите.
Ранг лексикографічний: **навчився → безпечно → оптимально → швидко**.

In [ ]:
print(f"{'seed':>5} {'L*':>7} {'L':>7} {'L*/L':>7}  дійшов")
for s in PUBLIC_SEEDS:
    e = GridWorld(N_TREES, s)
    Ls = optimal_length(e.trees)
    Qs, _ = train(GridWorld(N_TREES, s), seed=s)
    r = greedy_rollout(GridWorld(N_TREES, s), Qs)
    eff = Ls / r["length"] if r["success"] else 0.0
    print(f"{s:>5} {Ls:7.3f} {r['length']:7.3f} {eff:7.3f}  {r['success']}")